# Asistente Robótico por Comandos de Voz
## Exploración, Entrenamiento y Evaluación

**Universidad Rafael Landívar — Inteligencia Artificial — Primer Semestre 2026**

Este notebook es el entregable reproducible del proyecto. Ejecutar las celdas en orden genera:
- Análisis exploratorio del corpus de audio
- Visualizaciones de MFCC y Data Augmentation
- Entrenamiento del Modelo Base (CommandCNN) y Modelo Avanzado (CommandLSTM)
- Matrices de confusión y reporte completo de métricas
- Análisis comparativo entre modelos
- Análisis de latencia del pipeline completo


## 0. Configuración del Entorno

In [2]:
import sys, os
from pathlib import Path

# Agregar src/ al path para importar módulos del proyecto
REPO_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import tensorflow as tf
import keras
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

from model import (COMMANDS, NUM_CLASSES, SAMPLE_RATE, DURATION,
                   N_MFCC, N_FFT, HOP_LENGTH, TIME_FRAMES, INPUT_SHAPE,
                   build_cnn, build_lstm)
from train import (load_audio, extract_mfcc, load_dataset, split_dataset,
                   augment_time_shift, augment_pitch_shift,
                   augment_noise_injection, augment_time_stretch,
                   augment_spec_augment, DATASET_DIR, MODELS_DIR, DOCS_DIR)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 11
sns.set_theme(style="whitegrid")

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Librosa    : {librosa.__version__}")
print(f"Dataset    : {DATASET_DIR}")
print(f"Clases     : {COMMANDS}")


ModuleNotFoundError: No module named 'tensorflow'

## 1. Análisis Exploratorio del Corpus

In [ ]:
# Contar muestras originales por clase (sin augmentation)
class_counts = {}
all_files    = {}

for cmd in COMMANDS:
    wavs = list((DATASET_DIR / cmd).glob("*.wav"))
    class_counts[cmd] = len(wavs)
    all_files[cmd]    = wavs

total = sum(class_counts.values())
print("Distribución del corpus propio:")
print("-" * 40)
for cmd, n in class_counts.items():
    barra = "█" * (n // 5)
    print(f"  {cmd:15s}: {n:4d}  {barra}")
print("-" * 40)
print(f"  {'TOTAL':15s}: {total:4d} muestras")


In [ ]:
# Gráfica de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribución del Corpus de Audio", fontsize=14, fontweight="bold")

colors = ["#2196F3","#4CAF50","#FF9800","#E91E63","#9C27B0","#607D8B"]

# Barras
axes[0].bar(class_counts.keys(), class_counts.values(),
            color=colors, edgecolor="white", linewidth=1.2)
axes[0].set_title("Muestras por clase (originales)")
axes[0].set_ylabel("Cantidad de muestras")
axes[0].set_xlabel("Clase")
for i, (k, v) in enumerate(class_counts.items()):
    axes[0].text(i, v + 3, str(v), ha="center", fontweight="bold", fontsize=10)
axes[0].tick_params(axis="x", rotation=30)

# Pie
axes[1].pie(class_counts.values(), labels=class_counts.keys(),
            colors=colors, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1].set_title("Proporción por clase")

plt.tight_layout()
plt.savefig(DOCS_DIR / "distribucion_dataset.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Gráfica guardada en docs/distribucion_dataset.png")


## 2. Visualización de Audio y Features MFCC

In [ ]:
# Cargar una muestra de cada clase para visualizar
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(16, NUM_CLASSES * 2.8))
fig.suptitle("Forma de onda, Mel-Spectrogram y MFCC por clase",
             fontsize=14, fontweight="bold", y=1.01)

for i, cmd in enumerate(COMMANDS):
    wavs = all_files[cmd]
    if not wavs:
        continue
    audio = load_audio(str(wavs[0]))
    mfcc  = extract_mfcc(audio)
    mel   = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH,
                                           n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    times  = np.linspace(0, DURATION, len(audio))

    # Forma de onda
    axes[i, 0].plot(times, audio, color=colors[i], linewidth=0.8)
    axes[i, 0].set_ylabel(cmd, fontsize=10, fontweight="bold")
    axes[i, 0].set_ylim(-1.1, 1.1)
    if i == 0: axes[i, 0].set_title("Forma de onda", fontsize=11)
    if i == NUM_CLASSES - 1: axes[i, 0].set_xlabel("Tiempo (s)")

    # Mel-Spectrogram
    librosa.display.specshow(mel_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=axes[i, 1],
                             cmap="magma")
    if i == 0: axes[i, 1].set_title("Mel-Spectrogram (dB)", fontsize=11)
    axes[i, 1].set_ylabel("")

    # MFCC
    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis="time", ax=axes[i, 2], cmap="coolwarm")
    if i == 0: axes[i, 2].set_title(f"MFCC ({N_MFCC} coef.)", fontsize=11)
    axes[i, 2].set_ylabel("")

plt.tight_layout()
plt.savefig(DOCS_DIR / "audio_features_por_clase.png",
            dpi=150, bbox_inches="tight")
plt.show()


## 3. Data Augmentation — Visualización de Técnicas

In [ ]:
# Tomar una muestra de AVANZA para ilustrar todas las técnicas
sample_wav = all_files["AVANZA"][0]
audio_orig = load_audio(str(sample_wav))

augmented = {
    "Original"          : audio_orig,
    "Time Shift"        : augment_time_shift(audio_orig),
    "Pitch Shift"       : augment_pitch_shift(audio_orig),
    "Noise Injection"   : augment_noise_injection(audio_orig),
    "Time Stretch"      : augment_time_stretch(audio_orig),
}

fig, axes = plt.subplots(len(augmented), 2, figsize=(14, len(augmented)*2.5))
fig.suptitle("Técnicas de Data Augmentation — Clase AVANZA",
             fontsize=14, fontweight="bold")

for i, (nombre, audio) in enumerate(augmented.items()):
    mfcc = extract_mfcc(audio)
    times = np.linspace(0, DURATION, len(audio))

    # Forma de onda
    col = "#2196F3" if nombre == "Original" else "#FF9800"
    axes[i, 0].plot(times, audio, color=col, linewidth=0.9)
    axes[i, 0].set_ylabel(nombre, fontsize=10, fontweight="bold")
    axes[i, 0].set_ylim(-1.2, 1.2)
    if i == 0: axes[i, 0].set_title("Forma de onda")
    if i == len(augmented)-1: axes[i, 0].set_xlabel("Tiempo (s)")

    # MFCC resultante
    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis="time", ax=axes[i, 1], cmap="coolwarm")
    if i == 0: axes[i, 1].set_title("MFCC extraído")
    axes[i, 1].set_ylabel("")

plt.tight_layout()
plt.savefig(DOCS_DIR / "data_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()
print("5 técnicas de augmentation visualizadas.")


In [ ]:
# SpecAugment — mostrar antes y después sobre el MFCC
mfcc_orig = extract_mfcc(audio_orig)
mfcc_spec = augment_spec_augment(mfcc_orig)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("SpecAugment — Máscaras en frecuencia y tiempo",
             fontsize=13, fontweight="bold")

librosa.display.specshow(mfcc_orig, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                         x_axis="time", ax=axes[0], cmap="coolwarm")
axes[0].set_title("MFCC original")

librosa.display.specshow(mfcc_spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                         x_axis="time", ax=axes[1], cmap="coolwarm")
axes[1].set_title("MFCC con SpecAugment")

plt.tight_layout()
plt.savefig(DOCS_DIR / "spec_augment.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Carga del Dataset y División Train / Val / Test

In [ ]:
# Cargar todo el corpus con augmentation aplicada
X_cnn, X_lstm, y, class_names = load_dataset(augment=True)

# División 70 / 15 / 15
(X_cnn_train, X_cnn_val, X_cnn_test,
 X_lstm_train, X_lstm_val, X_lstm_test,
 y_train, y_val, y_test) = split_dataset(X_cnn, X_lstm, y)

print(f"\nForma de X_cnn  : {X_cnn.shape}   → (N, {N_MFCC}, {TIME_FRAMES}, 1)")
print(f"Forma de X_lstm : {X_lstm.shape}  → (N, {TIME_FRAMES}, {N_MFCC})")
print(f"Clases          : {class_names}")


## 5. Entrenamiento — Modelo Base (CommandCNN)

In [ ]:
from train import get_callbacks, EPOCHS_CNN, BATCH_SIZE, LEARNING_RATE

cnn_model = build_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()


In [ ]:
import time

print("Entrenando CommandCNN...")
t0 = time.time()
cnn_history = cnn_model.fit(
    X_cnn_train, y_train,
    validation_data=(X_cnn_val, y_val),
    epochs=EPOCHS_CNN,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks("cnn_nb"),
    verbose=1
)
print(f"\nTiempo de entrenamiento: {time.time()-t0:.1f}s")


In [ ]:
# Curvas de entrenamiento CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CommandCNN — Curvas de Entrenamiento", fontsize=14, fontweight="bold")

axes[0].plot(cnn_history.history["accuracy"],    label="Train",      color="#2196F3", lw=2)
axes[0].plot(cnn_history.history["val_accuracy"],label="Validación", color="#FF9800", lw=2, ls="--")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Época")
axes[0].set_ylabel("Accuracy"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(cnn_history.history["loss"],    label="Train",      color="#2196F3", lw=2)
axes[1].plot(cnn_history.history["val_loss"],label="Validación", color="#FF9800", lw=2, ls="--")
axes[1].set_title("Loss"); axes[1].set_xlabel("Época")
axes[1].set_ylabel("Loss"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_DIR / "training_curves_cnn_nb.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Evaluación — CommandCNN sobre Test Set

In [ ]:
y_pred_cnn   = np.argmax(cnn_model.predict(X_cnn_test, verbose=0), axis=1)
acc_cnn      = accuracy_score(y_test, y_pred_cnn)

print(f"Accuracy CNN en Test Set: {acc_cnn*100:.2f}%\n")
print(classification_report(y_test, y_pred_cnn,
                             target_names=class_names, digits=4))


In [ ]:
# Matriz de confusión CNN
cm_cnn = confusion_matrix(y_test, y_pred_cnn)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_cnn, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, linewidths=0.5)
ax.set_title(f"Matriz de Confusión — CommandCNN  (Accuracy: {acc_cnn*100:.2f}%)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Etiqueta Real"); ax.set_xlabel("Etiqueta Predicha")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(DOCS_DIR / "confusion_matrix_cnn_nb.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Entrenamiento — Modelo Avanzado (CommandLSTM)

In [ ]:
from train import EPOCHS_LSTM

lstm_model = build_lstm(time_steps=TIME_FRAMES, n_features=N_MFCC,
                        num_classes=NUM_CLASSES, use_gru=False)
lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
lstm_model.summary()


In [ ]:
print("Entrenando CommandLSTM...")
t0 = time.time()
lstm_history = lstm_model.fit(
    X_lstm_train, y_train,
    validation_data=(X_lstm_val, y_val),
    epochs=EPOCHS_LSTM,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks("lstm_nb"),
    verbose=1
)
print(f"\nTiempo de entrenamiento: {time.time()-t0:.1f}s")


In [ ]:
# Curvas de entrenamiento LSTM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CommandLSTM — Curvas de Entrenamiento", fontsize=14, fontweight="bold")

axes[0].plot(lstm_history.history["accuracy"],     label="Train",      color="#4CAF50", lw=2)
axes[0].plot(lstm_history.history["val_accuracy"], label="Validación", color="#FF5722", lw=2, ls="--")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Época")
axes[0].set_ylabel("Accuracy"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(lstm_history.history["loss"],     label="Train",      color="#4CAF50", lw=2)
axes[1].plot(lstm_history.history["val_loss"], label="Validación", color="#FF5722", lw=2, ls="--")
axes[1].set_title("Loss"); axes[1].set_xlabel("Época")
axes[1].set_ylabel("Loss"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_DIR / "training_curves_lstm_nb.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Evaluación — CommandLSTM sobre Test Set

In [ ]:
y_pred_lstm = np.argmax(lstm_model.predict(X_lstm_test, verbose=0), axis=1)
acc_lstm    = accuracy_score(y_test, y_pred_lstm)

print(f"Accuracy LSTM en Test Set: {acc_lstm*100:.2f}%\n")
print(classification_report(y_test, y_pred_lstm,
                             target_names=class_names, digits=4))


In [ ]:
# Matriz de confusión LSTM
cm_lstm = confusion_matrix(y_test, y_pred_lstm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_lstm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, linewidths=0.5)
ax.set_title(f"Matriz de Confusión — CommandLSTM  (Accuracy: {acc_lstm*100:.2f}%)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Etiqueta Real"); ax.set_xlabel("Etiqueta Predicha")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(DOCS_DIR / "confusion_matrix_lstm_nb.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Comparativa CNN vs LSTM

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def metricas(y_true, y_pred):
    return {
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-score" : f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

m_cnn  = metricas(y_test, y_pred_cnn)
m_lstm = metricas(y_test, y_pred_lstm)

print(f"{'Métrica':12s}  {'CommandCNN':>12s}  {'CommandLSTM':>12s}")
print("-" * 42)
for k in m_cnn:
    print(f"{k:12s}  {m_cnn[k]*100:11.2f}%  {m_lstm[k]*100:11.2f}%")


In [ ]:
# Gráfica comparativa de las 4 métricas
metricas_nombres = list(m_cnn.keys())
vals_cnn  = [m_cnn[k]*100  for k in metricas_nombres]
vals_lstm = [m_lstm[k]*100 for k in metricas_nombres]

x = np.arange(len(metricas_nombres))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, vals_cnn,  w, label="CommandCNN",  color="#2196F3", edgecolor="white")
b2 = ax.bar(x + w/2, vals_lstm, w, label="CommandLSTM", color="#4CAF50", edgecolor="white")

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom",
            fontsize=9, fontweight="bold")

ax.set_xticks(x); ax.set_xticklabels(metricas_nombres, fontsize=12)
ax.set_ylim(0, 115)
ax.set_ylabel("Porcentaje (%)", fontsize=12)
ax.set_title("Comparativa de Métricas — CNN vs LSTM", fontsize=14, fontweight="bold")
ax.axhline(80, color="red", ls="--", alpha=0.5, label="Umbral mínimo 80%")
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_DIR / "comparativa_modelos.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Análisis de Errores

In [ ]:
# Mostrar las confusiones más frecuentes del CNN
print("Confusiones más frecuentes — CommandCNN:")
print("-" * 50)
errores = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_cnn[i, j] > 0:
            errores.append((cm_cnn[i, j], class_names[i], class_names[j]))

errores.sort(reverse=True)
for n, real, pred in errores[:10]:
    print(f"  {real:15s} → predicho como {pred:15s}: {n} veces")

if not errores:
    print("  ¡Sin confusiones! Clasificación perfecta en test set.")


## 11. Exportar Modelos

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)

cnn_model.save(str(MODELS_DIR  / "command_cnn.keras"))
lstm_model.save(str(MODELS_DIR / "command_lstm.keras"))
cnn_model.save(str(MODELS_DIR  / "command_cnn.h5"))
lstm_model.save(str(MODELS_DIR / "command_lstm.h5"))

print("Modelos guardados:")
for f in MODELS_DIR.glob("command_*"):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:30s}  {size_mb:.2f} MB")


## 12. Análisis de Latencia del Pipeline

In [ ]:
import time as _time

# Medir latencia de cada componente con 50 muestras reales del test set
N_BENCH = min(50, len(X_cnn_test))

lat_mfcc, lat_cnn, lat_lstm = [], [], []

for i in range(N_BENCH):
    # Simular extracción MFCC desde audio crudo
    raw = np.random.randn(SAMPLE_RATE).astype(np.float32)

    t0 = _time.perf_counter()
    _ = extract_mfcc(raw)
    lat_mfcc.append((_time.perf_counter() - t0) * 1000)

    # Inferencia CNN
    sample_cnn = X_cnn_test[i:i+1]
    t0 = _time.perf_counter()
    _ = cnn_model.predict(sample_cnn, verbose=0)
    lat_cnn.append((_time.perf_counter() - t0) * 1000)

    # Inferencia LSTM
    sample_lstm = X_lstm_test[i:i+1]
    t0 = _time.perf_counter()
    _ = lstm_model.predict(sample_lstm, verbose=0)
    lat_lstm.append((_time.perf_counter() - t0) * 1000)

# Estimación WiFi (medición representativa, sin hardware)
lat_wifi_est = 8.0   # ms típico en red local

print(f"{'Componente':25s} {'Media':>9s} {'Mín':>9s} {'Máx':>9s}")
print("-" * 58)
for nombre, lats in [("Extracción MFCC", lat_mfcc),
                     ("Inferencia CNN",  lat_cnn),
                     ("Inferencia LSTM", lat_lstm)]:
    print(f"  {nombre:23s} {np.mean(lats):7.1f}ms  {np.min(lats):7.1f}ms  {np.max(lats):7.1f}ms")

total_est = np.mean(lat_mfcc) + np.mean(lat_cnn) + lat_wifi_est
print(f"\n  {'Envío WiFi (estimado)':23s} {lat_wifi_est:7.1f}ms")
print(f"  {'TOTAL estimado':23s} {total_est:7.1f}ms  {'✅ < 500ms' if total_est < 500 else '⚠️ > 500ms'}")


In [ ]:
# Gráfica de latencia por componente
componentes = ["Extracción\nMFCC", "Inferencia\nCNN", "Inferencia\nLSTM", "WiFi\n(estimado)"]
medias = [np.mean(lat_mfcc), np.mean(lat_cnn), np.mean(lat_lstm), lat_wifi_est]
colores = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análisis de Latencia del Pipeline", fontsize=14, fontweight="bold")

# Barras de media
bars = axes[0].bar(componentes, medias, color=colores, edgecolor="white", linewidth=1.2)
for bar, v in zip(bars, medias):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{v:.1f}ms", ha="center", fontweight="bold", fontsize=10)
axes[0].axhline(500, color="red", ls="--", alpha=0.5, label="Límite 500ms")
axes[0].set_ylabel("Latencia (ms)"); axes[0].set_title("Latencia media por componente")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

# Boxplot de MFCC y CNN
axes[1].boxplot([lat_mfcc, lat_cnn, lat_lstm],
                labels=["MFCC", "CNN", "LSTM"],
                patch_artist=True,
                boxprops=dict(facecolor="#E3F2FD"),
                medianprops=dict(color="#1565C0", linewidth=2))
axes[1].set_ylabel("Latencia (ms)"); axes[1].set_title("Distribución de latencias")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_DIR / "latencia_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. Resumen Final

In [ ]:
print("=" * 55)
print("  RESUMEN FINAL DEL PROYECTO")
print("=" * 55)
print(f"  Dataset total (con augmentation) : {len(y):,} muestras")
print(f"  Train / Val / Test               : 70% / 15% / 15%")
print()
print(f"  CommandCNN  — Accuracy test  : {acc_cnn*100:.2f}%")
print(f"  CommandLSTM — Accuracy test  : {acc_lstm*100:.2f}%")
print()

archivos = list(DOCS_DIR.glob("*.png")) + list(MODELS_DIR.glob("command_*"))
print(f"  Archivos generados: {len(archivos)}")
for f in sorted(archivos):
    print(f"    {f.relative_to(f.parent.parent)}")
print()
print("✅ Notebook completado.")
